# BIOSTAT 707 — Checkpoint 2
## Predicting in-hospital death

**Author:** Siyu Gong  
**NetID:** sg763

### AI-use statement

**Artificial intelligence (AI) use:** I used OpenAI Codex to assist with code implementation, debugging, report writing and checking the to do list. I specified the analysis choices, including data cleaning, imputation, and model selection. Codex helped with efficiency, but its not always give right suggestion when have multiple choices. Automated checks cover matching IDs, fold separation, saved predictions and reproducible execution. I remain responsible for reviewing and understanding the analysis.


### 1. Setup

These results use the locked Pixi environment. Open the notebook with `pixi run lab` to use the same package versions as the grading command.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time
import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib
from IPython.display import display
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score, roc_curve, log_loss, brier_score_loss
from threadpoolctl import threadpool_limits
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

SEED = 707
CATEGORICAL = ['Gender', 'ICUType']
START = time.perf_counter()
np.random.seed(SEED)
thread_limit = threadpool_limits(limits=2)
sns.set_theme(style='whitegrid', font_scale=1.0)
plt.rcParams.update({'figure.dpi':120, 'savefig.dpi':170})

### 2. Read the data

Use `output/set-a_wide.csv` from Checkpoint 1. Each row is one intensive care unit (ICU) stay from 00:00-48:00.


- **Imput data:** cleaned wide table from Checkpoint 1, `output/set-a_wide.csv`; 222 predictors (`X`) and one outcome (`y`).

- **Outcome:** `In-hospital_death` from `Outcomes-a.txt`; 0 = survived, 1 = died; Joined by `RecordID` in Checkpoint 1.

- **Data layout:** one row per ICU stay; admission information and 0–48-hour measurement summaries; 4,000 rows with unique RecordIDs.

- **Cross-validation grouping:** one row per stay; all information from the same stay remains together; stratified five-fold cross-validation (CV).

- **Prediction per stay:** one out-of-fold death probability per stay; direct comparison with the observed outcome, Simplified Acute Physiology Score I (SAPS-I) and Sequential Organ Failure Assessment (SOFA).

- **Cohort size:** 4,000 ICU stays; 554 deaths; mortality rate = 13.85%.

- **Cleaning decisions:**  −1 and impossible values, including zero height or weight, treated as NaN; summaries calculated from valid measurements.

- **Data imputation:** numeric missing values filled with training-fold column means; categorical missing values filled with training-fold modes; counts and missingness indicators retained; inputs standardized after imputation using training-fold means and standard deviations.

**Data source:** Silva et al. (2012), *Predicting In-Hospital Mortality of Patients in ICU: The PhysioNet/Computing in Cardiology Challenge 2012*. [PhysioNet Challenge 2012, version 1.0.0](https://physionet.org/content/challenge-2012/1.0.0/), set-a; Open Data Commons Attribution License v1.0.


In [ ]:
wide = pd.read_csv('output/set-a_wide.csv', index_col='RecordID')
assert len(wide) == 4000 and wide.index.is_unique
outcome_columns = ['SAPS-I', 'SOFA', 'Length_of_stay', 'Survival', 'In-hospital_death']
outcomes = wide[outcome_columns]
y = outcomes['In-hospital_death'].astype(int)
assert y.isin([0, 1]).all()

variables = sorted(c.removesuffix('_count') for c in wide
                   if c.endswith('_count') and c != 'MechVent_count')
predictors = ['Age', 'Gender', 'Height', 'ICUType', 'AdmissionWeight']
predictors += [f'{v}_{s}' for s in ['count', 'last', 'min', 'max', 'mean'] for v in variables]
predictors += [f'{v}_missing' for v in variables]
X = wide[predictors].copy()
X['MechVent_documented'] = wide['MechVent_max'].eq(1).astype(int)
predictors = X.columns.tolist()
assert X.shape == (4000, 222)
assert X.filter(regex='_count$|_missing$|^MechVent_documented$').notna().all().all()
assert not set(outcome_columns + ['RecordID']) & set(predictors)
X.join(y).to_csv('output/cp2_modeling_table.csv')


In [ ]:
cohort = pd.DataFrame({'Stays':[len(y)], 'Deaths':[y.sum()], 'Deaths (%)':[100*y.mean()],
                      'Candidate predictors':[len(predictors)]}, index=['set-a'])
display(cohort.style.format(precision=2, na_rep="—")
    .set_caption('Table 1. Modeling cohort'))

Table 1 shows 4,000 ICU stays and 554 deaths (13.85%).

In [ ]:
display(wide[['Age', 'Gender', 'ICUType', 'HR_mean', 'Lactate_mean', 'In-hospital_death']].head(3).style.format(precision=2, na_rep="NA")
    .set_caption('Table 2. Example rows and columns for wide table'))

Table 2 shows example rows of the cleaned wide table. Each row has one outcome and its own predictors; missing measurements remain NaN before fold-specific imputation.

### 3. Predictors

I predict death using information from the first 48 hours.

- Exclude RecordID, Length_of_stay, Survival and In-hospital_death.
- Use Simplified Acute Physiology Score I (SAPS-I) and Sequential Organ Failure Assessment (SOFA) only as benchmarks.
- Keep measurement counts and missingness indicators because testing may reflect illness severity.
- MechVent_documented is 1 if ventilation is recorded. A 0 means no documentation.

#### Variable names

ALP = alkaline phosphatase;

ALT/AST = alanine/aspartate aminotransferase;

BUN = blood urea nitrogen;

DiasABP/SysABP = diastolic/systolic invasive arterial blood pressure;

MAP = mean arterial pressure;

the NI prefix means noninvasive;

FiO2 = fraction of inspired oxygen;

GCS = Glasgow Coma Scale;

HCO3 = bicarbonate;

HCT = hematocrit;

HR = heart rate;

K/Mg/Na = potassium/magnesium/sodium;

MechVent = mechanical ventilation;

PaCO2/PaO2 = arterial partial pressure of carbon dioxide/oxygen;

RespRate = respiratory rate;

SaO2 = arterial oxygen saturation;

Temp = temperature;

WBC = white blood cell count. TroponinI and TroponinT denote troponin I and T.

No predictors are dropped solely for high missingness, so no missingness cutoff is used. Table 3b lists the excluded columns and design choices.


**Table 3. Predictor inventory**

| Group | Variables | Count | Reason |
|---|---|---:|---|
| Admission descriptors | Age, Height, AdmissionWeight, Gender, ICUType | 5 | Baseline differences in age, sex and body size; ICU type reflects admission case mix |
| 48-hour mean | ALP, ALT, AST, Albumin, BUN, Bilirubin, Cholesterol, Creatinine, DiasABP, FiO2, GCS, Glucose, HCO3, HCT, HR, K, Lactate, MAP, Mg, NIDiasABP, NIMAP, NISysABP, Na, PaCO2, PaO2, Platelets, RespRate, SaO2, SysABP, Temp, TroponinI, TroponinT, Urine, WBC, Weight, pH | 36 | Average cardiovascular, respiratory, neurological and organ-function status |
| 48-hour minimum and maximum | The same variables | 72 | Episodes of hypotension, hypoxemia or marked laboratory abnormalities |
| Last available value | The same variables | 36 | Most recent recorded clinical condition before the prediction time |
| Valid measurement count | The same variables | 36 | Monitoring intensity may reflect instability, suspected complications or routine protocols |
| No valid measurement indicator | The same variables | 36 | Selective testing may reflect clinical concern; incomplete recording can also cause missingness |
| Ventilation documentation | MechVent_documented | 1 | Respiratory support or airway protection, including perioperative ventilation |
| **Total** | | **222** | |


Table 3 groups the 222 inputs by clinical role. The counts and missingness indicators preserve information about testing intensity.

**Table 3b. Excluded columns and rules**

| Group | Excluded columns | Reason | Threshold |
|---|---|---|---|
| Identifier | RecordID | Not a predictor | Not applicable |
| Outcome | In-hospital_death | Target only | Not applicable |
| Later outcomes | Length_of_stay, Survival | After the prediction window | Not applicable |
| Benchmark scores | SAPS-I, SOFA | Comparison only | Not applicable |
| First values | ALP_first, ALT_first, AST_first, Albumin_first, BUN_first, Bilirubin_first, Cholesterol_first, Creatinine_first, DiasABP_first, FiO2_first, GCS_first, Glucose_first, HCO3_first, HCT_first, HR_first, K_first, Lactate_first, MAP_first, MechVent_first, Mg_first, NIDiasABP_first, NIMAP_first, NISysABP_first, Na_first, PaCO2_first, PaO2_first, Platelets_first, RespRate_first, SaO2_first, SysABP_first, Temp_first, TroponinI_first, TroponinT_first, Urine_first, WBC_first, Weight_first, pH_first | Use mean, min, max and last summaries to limit repeated information | No numeric cutoff |
| Ventilation summaries | MechVent_count, MechVent_last, MechVent_min, MechVent_max, MechVent_mean, MechVent_missing | Replace with MechVent_documented; 1 if MechVent_max equals 1 | Recorded code = 1 |


Table 3b lists the columns excluded from modeling. No feature is removed because it exceeds a missingness percentage.

In [ ]:
model_inventory = pd.DataFrame([
    ['Ridge', len(predictors), 'All candidate inputs; L2-penalized logistic regression'],
    ['Splines', 15, 'Top 15 original inputs by absolute standardized ridge coefficient'],
    ['Random forest', len(predictors), 'All candidate inputs; random forest'],
], columns=['Model', 'Input count', 'Rule']).set_index('Model')
display(model_inventory.style.format(precision=0, na_rep="—")
    .set_caption('Table 4. Model inputs before category encoding or spline expansion'))

Table 4 shows 222 original inputs for ridge and random forest, and 15 for splines. One-hot encoding and spline expansion can increase the number of fitted terms.

### 4. Cross-validation

- Use 5 stratified folds, with seed 707. All models and benchmarks use the same folds.
- Choose hyperparameters by the lowest mean CV log loss. No nested CV.
- Numeric missing values: training-fold mean. Categories: training-fold mode, then one-hot encoding.
- After imputation and encoding, standardize all model inputs using the training-fold mean and standard deviation. For splines, expand the imputed measurements before standardization.
- Select the spline model's 15 inputs from a ridge fit within each training fold. Use the C selected for Model 1; no extra CV inside the selector.
- Fit imputation, encoding, standardization, spline knots and feature selection on training stays only, then apply them to validation stays.

**Optimism:** I use the same five folds to select hyperparameters and report performance. The selected C is also reused for ridge feature selection. These choices use validation results, so the reported CV scores may be optimistic.


Use mean imputation because it is simple and keeps every stay in the analysis. Modes preserve valid category codes. Missingness indicators and counts retain information about which tests were performed. Mean imputation reduces variation and does not represent imputation uncertainty.

AUROC means area under the receiver operating characteristic curve; SD means standard deviation.


In [ ]:
names = ['Ridge','Splines','Random forest']
folds = list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(X,y))
oof = pd.DataFrame(index=X.index,columns=names,dtype=float)
oof['fold'] = 0
for fold,(train,test) in enumerate(folds,1):
    assert not set(X.index[train]) & set(X.index[test])
    oof.loc[X.index[test],'fold'] = fold
assert oof['fold'].between(1,5).all()
full_models, full_parameters = {}, {}
fold_rows, tune_rows = [], []

In [ ]:
fold_summary = pd.DataFrame([{'Fold':i,'Training stays':len(tr),'Held-out stays':len(te),
                             'Held-out deaths':int(y.iloc[te].sum())} for i,(tr,te) in enumerate(folds,1)])
display(fold_summary.set_index('Fold').style.format(precision=0, na_rep="—")
    .set_caption('Table 5. Shared folds'))

Table 5 shows 3,200 training stays and 800 validation stays in each fold, with 110 or 111 validation deaths.

### 5. Model 1: ridge regression

I use ridge logistic regression with all 222 predictors. Many summaries are correlated, so an L2 penalty helps shrink their coefficients.

Table 6a lists the tuning grid. Smaller C means stronger shrinkage.

After choosing parameters by five-fold CV, refit each model on all of set-a for interpretation and saving. The reported CV predictions use the same selected parameters, with each model refit on its training fold. Figure 1 shows the 25 largest absolute ridge coefficients, measured per one standard deviation of the transformed predictor.

#### Define ridge and preprocessing

In [ ]:
def ridge_importance(ridge):
    """Absolute standardized coefficients, grouped back to original inputs."""
    names = ridge.named_steps['preprocess'].get_feature_names_out()
    scores = pd.Series(np.abs(ridge.named_steps['model'].coef_[0]), index=names)
    original = [next((c for c in CATEGORICAL if n.startswith(c + '_')), n) for n in names]
    return scores.groupby(original, sort=False).max().sort_values(ascending=False, kind='stable')

In [ ]:
binary = [c for c in predictors if c.endswith('_missing') or c == 'MechVent_documented']
counts = [c for c in predictors if c.endswith('_count')]
continuous = [c for c in predictors if c not in CATEGORICAL + binary + counts]
numeric = Pipeline([('impute',SimpleImputer(strategy='mean',keep_empty_features=True))])
categories = Pipeline([
    ('impute',SimpleImputer(strategy='most_frequent',keep_empty_features=True)),
    ('encode',OneHotEncoder(handle_unknown='ignore',drop='first',sparse_output=False))
])
preprocess = ColumnTransformer([
    ('numeric',numeric,continuous),
    ('category',categories,CATEGORICAL),
    ('availability','passthrough',binary + counts)
],remainder='drop',verbose_feature_names_out=False)
ridge_pipe = Pipeline([
    ('preprocess',preprocess),
    ('scale',StandardScaler()),
    ('model',LogisticRegression(C=1,l1_ratio=0,solver='lbfgs',max_iter=3000,random_state=SEED))
])

#### Tuning grid

**Table 6a. Ridge tuning grid**

| Parameter | Values | Role |
|---|---|---|
| `C` | 0.01, 0.1, 1, 10 | Tuned; inverse L2 penalty strength |
| Penalty | L2 (ridge) | Fixed |
| Solver / maximum iterations | lbfgs / 3,000 | Fixed |

Table 6a gives four candidate settings. I select the lowest mean five-fold log loss.

#### Tune ridge


In [ ]:
ridge_grid = {'model__C':[0.01,0.1,1.0,10.0]}
search = GridSearchCV(ridge_pipe,ridge_grid,scoring='neg_log_loss',cv=folds,n_jobs=1,error_score='raise')
search.fit(X,y)
full_models['Ridge'] = search.best_estimator_
full_parameters['Ridge'] = search.best_params_
ridge_tuning = pd.DataFrame([
    {'Model':'Ridge', 'Parameters':str(params), 'CV log loss':-score,
     'Selected':params == search.best_params_}
    for params,score in zip(search.cv_results_['params'],search.cv_results_['mean_test_score'])
])
tune_rows.extend(ridge_tuning.to_dict('records'))
display(ridge_tuning.drop(columns='Model').style.format(precision=4)
    .set_caption('Table 6b. Ridge tuning results'))

Table 6b selects C = 0.01 by the lowest mean CV log loss.


#### Ridge predictions by fold

In [ ]:
mean_rows, ridge_fold_features = [], {}
for fold,(train,test) in enumerate(folds,1):
    model = clone(full_models['Ridge']).fit(X.iloc[train],y.iloc[train])
    imputer = model.named_steps['preprocess'].named_transformers_['numeric'].named_steps['impute']
    mean_rows.extend({'Fold':fold, 'Feature':v, 'Mean':m}
        for v,m in zip(continuous,imputer.statistics_))
    ridge_fold_features[fold] = ridge_importance(model).head(15).index.tolist()
    pred = model.predict_proba(X.iloc[test])[:,1]
    oof.loc[X.index[test],'Ridge'] = pred
    fold_rows.append({'Model':'Ridge','Fold':fold,'AUROC':roc_auc_score(y.iloc[test],pred),
                      'Log loss':log_loss(y.iloc[test],pred),'Brier':brier_score_loss(y.iloc[test],pred)})
pd.DataFrame(mean_rows).to_csv('output/cp2_fold_imputation_means.csv',index=False)

In [ ]:
ridge_scores = pd.DataFrame(fold_rows).query("Model == 'Ridge'").set_index('Fold').drop(columns='Model')
ridge_report = ridge_scores.copy()
ridge_report.loc['Mean'] = ridge_scores.mean()
ridge_report.loc['SD'] = ridge_scores.std(ddof=1)
display(ridge_report.style.format(precision=3)
    .set_caption('Table 6c. Ridge fold scores, mean and SD'))

Table 6c reports all five ridge fold scores. Mean AUROC is 0.842 (SD 0.018). Log loss and Brier score summarize prediction errors.


#### Ridge coefficients

In [ ]:
ridge = full_models['Ridge']
imputer = ridge.named_steps['preprocess'].named_transformers_['numeric'].named_steps['impute']
feature_means = pd.Series(imputer.statistics_, index=continuous, name='Mean')
feature_means.index.name = 'Feature'
feature_means.to_csv('output/cp2_imputation_means.csv')
display(feature_means.head(10).to_frame().style.format(precision=3, na_rep="—")
    .set_caption('Table 6d. First 10 full-data imputation means'))

- Table 6d shows the first 10 full-data imputation means. Cross-validation uses each training fold's own means.

In [ ]:
coef = pd.Series(ridge.named_steps['model'].coef_[0],
                 index=ridge.named_steps['preprocess'].get_feature_names_out(),name='Coefficient')
coef.to_csv('output/cp2_ridge_coefficients.csv')
importance = ridge_importance(ridge)
importance.rename('Absolute standardized coefficient').to_csv('output/cp2_ridge_importance.csv')
top = coef.loc[coef.abs().nlargest(25).index].sort_values()
fig,ax = plt.subplots(figsize=(8.5,8))
ax.barh(top.index,top.values,color=['#bc553d' if v>0 else '#2b718c' for v in top.values])
ax.axvline(0,color='black',lw=.8)
ax.set(xlabel='Log-odds coefficient per 1 SD of transformed predictor',
       title='Figure 1. Largest ridge coefficients')
fig.tight_layout()
fig.savefig('output/cp2_ridge_coefficients.png', bbox_inches="tight")
plt.show()
plt.close(fig)

- Figure 1 shows Age has the largest positive coefficient (0.301), and GCS_last has the largest negative coefficient (−0.479). These are adjusted associations, not causal effects. The selected C is 0.01.


In [ ]:
ordered_coef = coef.loc[coef.abs().sort_values(ascending=False).index]
fig,axes = plt.subplots(1,4,figsize=(20,16),sharex=True)
for ax,positions in zip(axes,np.array_split(np.arange(len(ordered_coef)),4)):
    part = ordered_coef.iloc[positions]
    ax.barh(part.index,part.values,color=['#bc553d' if v>0 else '#2b718c' for v in part.values])
    ax.axvline(0,color='black',lw=.8)
    ax.invert_yaxis()
    ax.tick_params(axis='y',labelsize=8)
    ax.set_xlabel('Standardized coefficient')
fig.suptitle('Figure 1b. All standardized ridge coefficients')
fig.tight_layout()
fig.savefig('output/cp2_ridge_coefficients_all.png',bbox_inches='tight')
plt.show()
plt.close(fig)

Figure 1b includes every fitted ridge coefficient

### 6. Model 2: restricted cubic splines

I use the 15 inputs with the largest absolute standardized ridge coefficients. For a categorical variable, I use its largest absolute dummy coefficient and keep the whole variable. Selection is repeated within each training split.

- Continuous measurements: restricted cubic splines with knots at the training 5th, 27.5th, 50th, 72.5th and 95th percentiles. The tails are linear.
- Counts: linear terms. Missingness indicators: binary terms. Categories: one-hot encoding.
- L2 penalty: the four C values listed in Table 7a.
- Feature selection uses Model 1's selected C; the ranking is fitted again within each training fold.

Figure 2 shows odds ratios (ORs) relative to each variable's median. The median is only the plot reference; imputation still uses means. Curves cover the observed 5th–95th percentiles. The approximate pointwise 95% bands use the penalized Hessian.


#### Select 15 predictors and define splines

In [ ]:
def categorical_columns(X):
    return [c for c in X if c in CATEGORICAL]

def availability_columns(X):
    return [c for c in X if c.endswith(('_missing', '_count')) or c == 'MechVent_documented']

def continuous_columns(X):
    return [c for c in X if c not in categorical_columns(X) + availability_columns(X)]

In [ ]:
class RidgeTopFeatures(TransformerMixin, BaseEstimator):
    """Fit ridge and rank inputs using the training data supplied to fit."""
    def __init__(self, n_features=15, C=0.01):
        self.n_features, self.C = n_features, C
    def fit(self, X, y):
        ridge = clone(ridge_pipe)
        ridge.set_params(model__C=self.C).fit(X, y)
        self.ridge_C_ = self.C
        self.importance_ = ridge_importance(ridge)
        self.selected_features_ = self.importance_.head(self.n_features).index.tolist()
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.n_features_in_ = X.shape[1]
        return self
    def transform(self, X):
        return X.loc[:, self.selected_features_]
    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.selected_features_, dtype=object)

In [ ]:
class RestrictedCubicSpline(TransformerMixin, BaseEstimator):
    """Five quantile knots; cubic between knots and linear beyond boundary knots."""
    def fit(self, X, y=None):
        X = np.asarray(X)
        self.knots_ = [np.unique(np.quantile(X[:, j], [.05,.275,.5,.725,.95]))
                       for j in range(X.shape[1])]
        return self
    def transform(self, X):
        X = np.asarray(X)
        columns = []
        for j, k in enumerate(self.knots_):
            x = X[:, j]
            columns.append(x)
            if len(k) < 3:
                continue
            for knot in k[:-2]:
                h = (np.maximum(x-knot,0)**3
                     - np.maximum(x-k[-2],0)**3 * (k[-1]-knot)/(k[-1]-k[-2])
                     + np.maximum(x-k[-1],0)**3 * (k[-2]-knot)/(k[-1]-k[-2]))
                columns.append(h / (k[-1]-k[0])**2)
        return np.column_stack(columns)
    def get_feature_names_out(self, input_features=None):
        names=[]
        for name,k in zip(input_features,self.knots_):
            names.append(str(name))
            names.extend(f'{name}_rcs{i+1}' for i in range(max(0,len(k)-2)))
        return np.asarray(names,dtype=object)


In [ ]:
spline_pipe = Pipeline([
    ('select',RidgeTopFeatures(C=full_parameters['Ridge']['model__C'])),
    ('preprocess',ColumnTransformer([
        ('smooth',Pipeline([
            ('impute',SimpleImputer(strategy='mean',keep_empty_features=True)),
            ('spline',RestrictedCubicSpline())
        ]),continuous_columns),
        ('category',clone(categories),categorical_columns),
        ('availability','passthrough',availability_columns)
    ],remainder='drop',verbose_feature_names_out=False)),
    ('scale',StandardScaler()),
    ('model',LogisticRegression(C=1,l1_ratio=0,solver='lbfgs',max_iter=3000,random_state=SEED))
])

#### Tuning grid

**Table 7a. Spline tuning grid**

| Parameter | Values | Role |
|---|---|---|
| Logistic regression `C` | 0.01, 0.1, 1, 10 | Tuned |
| Number of original predictors | 15 | Fixed |
| Knot percentiles | 5, 27.5, 50, 72.5, 95 | Fixed; locations fitted on training stays |
| Ridge selector `C` | Model 1 selected C | Reused; no additional CV |
| Solver / maximum iterations | lbfgs / 3,000 | Fixed |

Table 7a gives four candidate penalty settings. I select the lowest mean five-fold log loss.

#### Tune splines


In [ ]:
splines_grid = {'model__C':[0.01,0.1,1.0,10.0]}
search = GridSearchCV(spline_pipe,splines_grid,scoring='neg_log_loss',cv=folds,n_jobs=1,error_score='raise')
search.fit(X,y)
full_models['Splines'] = search.best_estimator_
full_parameters['Splines'] = search.best_params_
splines_tuning = pd.DataFrame([
    {'Model':'Splines', 'Parameters':str(params), 'CV log loss':-score,
     'Selected':params == search.best_params_}
    for params,score in zip(search.cv_results_['params'],search.cv_results_['mean_test_score'])
])
tune_rows.extend(splines_tuning.to_dict('records'))
display(splines_tuning.drop(columns='Model').style.format(precision=4)
    .set_caption('Table 7b. Spline tuning results'))

Table 7b selects C = 1 by the lowest mean CV log loss.


#### Spline predictions by fold

In [ ]:
selection_rows = []
for fold,(train,test) in enumerate(folds,1):
    model = clone(full_models['Splines']).fit(X.iloc[train],y.iloc[train])
    selected = model.named_steps['select']
    assert selected.selected_features_ == ridge_fold_features[fold]
    selection_rows.extend({'Fold':fold,'Rank':rank,'Feature':feature,
                           'Ridge importance':selected.importance_[feature],
                           'Ridge C':selected.ridge_C_}
                          for rank,feature in enumerate(selected.selected_features_,1))
    pred = model.predict_proba(X.iloc[test])[:,1]
    oof.loc[X.index[test],'Splines'] = pred
    fold_rows.append({'Model':'Splines','Fold':fold,'AUROC':roc_auc_score(y.iloc[test],pred),
                      'Log loss':log_loss(y.iloc[test],pred),'Brier':brier_score_loss(y.iloc[test],pred)})
fold_selection = pd.DataFrame(selection_rows)
fold_selection.to_csv('output/cp2_spline_fold_features.csv',index=False)

In [ ]:
spline_scores = pd.DataFrame(fold_rows).query("Model == 'Splines'").set_index('Fold').drop(columns='Model')
spline_report = spline_scores.copy()
spline_report.loc['Mean'] = spline_scores.mean()
spline_report.loc['SD'] = spline_scores.std(ddof=1)
display(spline_report.style.format(precision=3)
    .set_caption('Table 7c. Spline fold scores, mean and SD'))

Table 7c reports all five spline fold scores. Mean AUROC is 0.823 (SD 0.020). The reported scores reuse the folds used to select the penalty.


#### Selected predictors and effects

In [ ]:
spline_model = full_models['Splines']
selected_features = spline_model.named_steps['select'].selected_features_
assert selected_features == importance.head(15).index.tolist()
selected_table = importance.loc[selected_features].rename('Ridge importance').to_frame()
selected_table.insert(0,'Rank',np.arange(1,16))
selected_table['CV folds selected (of 5)'] = fold_selection.Feature.value_counts().reindex(selected_features,fill_value=0)
selected_table.index.name = 'Feature'
selected_table.to_csv('output/cp2_spline_selected_features.csv')
display(selected_table.style.format(precision=3, na_rep="—")
    .set_caption('Table 7d. Fifteen inputs selected from the full-data ridge fit'))

In [ ]:
Z = spline_model[:-1].transform(X)
Z1 = np.column_stack([np.ones(len(Z)),Z])
p = spline_model.predict_proba(X)[:,1]
beta = np.r_[spline_model[-1].intercept_,spline_model[-1].coef_.ravel()]
penalty = np.diag(np.r_[0,np.repeat(1/spline_model[-1].C,Z.shape[1])])
cov = np.linalg.pinv((Z1.T * (p*(1-p))) @ Z1 + penalty)
base = X.median().to_frame().T
for cat in ['Gender','ICUType']:
    base[cat] = X[cat].mode().iloc[0]
# Plot the highest-ranked spline inputs from three distinct clinical measurements.
plot_vars, measurements = [], set()
for feature in continuous_columns(X[selected_features]):
    measurement = feature.split('_')[0]
    if measurement not in measurements:
        plot_vars.append(feature)
        measurements.add(measurement)
    if len(plot_vars) == 3:
        break
assert len(plot_vars) == 3
labels = [{'GCS_last':'Last GCS (points)', 'Age':'Age (years)',
           'pH_min':'Minimum arterial pH'}.get(feature,feature) for feature in plot_vars]
effect_rows = []
fig,axes = plt.subplots(1,3,figsize=(13.5,4))
for ax,var,label in zip(axes,plot_vars,labels):
    low,high = X[var].quantile([.05,.95])
    reference = X[var].median()
    grid = np.linspace(low,high,120)
    frame = pd.concat([base]*len(grid),ignore_index=True)
    frame[var] = grid
    ref = base.copy()
    ref[var] = reference
    contrast = spline_model[1:-1].transform(frame[selected_features]) - spline_model[1:-1].transform(ref[selected_features])
    contrast = np.column_stack([np.zeros(len(grid)),contrast])
    effect = contrast @ beta
    se = np.sqrt(np.maximum(0,np.einsum('ij,jk,ik->i',contrast,cov,contrast)))
    ax.plot(grid,np.exp(effect),color='#246887')
    ax.fill_between(grid,np.exp(effect-1.96*se),np.exp(effect+1.96*se),color='#246887',alpha=.2)
    ax.axhline(1,color='gray',ls='--',lw=1)
    rug = X[var].dropna()
    rug = rug[rug.between(low, high)]
    ax.plot(rug,np.repeat(.02,len(rug)),'|',transform=ax.get_xaxis_transform(),alpha=.06,color='black')
    ax.set(xlabel=label,ylabel='Adjusted OR versus median',title=f'Reference = {reference:.2f}')
    for xx,ee,ss in zip(grid,effect,se):
        effect_rows.append({'Variable':var,'Value':xx,'Reference':reference,'Log OR':ee,'SE':ss})
fig.suptitle('Figure 2. Restricted cubic spline effects with approximate pointwise 95% bands',y=1.04)
fig.tight_layout()
fig.savefig('output/cp2_spline_effects.png', bbox_inches="tight")
plt.show()
plt.close(fig)
pd.DataFrame(effect_rows).to_csv('output/cp2_spline_effects.csv',index=False)


Table 7d lists the final 15 inputs and how often each was selected across the five folds. Some inputs change across folds.

Figure 2 shows three effects:

- **Last GCS:** lower scores have higher predicted odds of death, with modest curvature. The OR at GCS 4 is 16.44 relative to GCS 15. This agrees with worse neurological status carrying higher risk.
- **Age:** the effect increases more steeply at older ages. The OR is 0.43 at age 29 and 2.16 at age 89, relative to age 67. This direction is clinically plausible.
- **Mean pH:** the curve is nonlinear, with higher odds at its upper end. At the plotted upper end (pH 7.47), the OR is 2.37 relative to 7.39. The flat lower end is less straightforward and may reflect adjustment for minimum pH.

These are adjusted associations. A curved OR plot alone does not prove nonlinearity on the log-odds scale.

### 7. Model 3: random forest

I use all 222 inputs. A random forest averages many decision trees and can capture nonlinear effects and interactions.

- Trees: 300; bootstrap sampling: on; split criterion: Gini impurity.
- Tune maximum depth and minimum samples per leaf, as listed in Table 8a.
- Missing values: training-fold numeric means and categorical modes; no native missing-value handling.
- Standardize inputs after imputation and encoding, using training-fold statistics, to keep the same preprocessing as Model 1.
- Early stopping: not used.

#### Define forest

In [ ]:
forest_pipe = Pipeline([
    ('preprocess',clone(preprocess)),
    ('scale',StandardScaler()),
    ('model',RandomForestClassifier(n_estimators=300,max_features='sqrt',
        criterion='gini',bootstrap=True,n_jobs=2,random_state=SEED))
])


#### Tuning grid

**Table 8a. Random forest tuning grid**

| Parameter | Values | Role |
|---|---|---|
| `max_depth` | 10, None (unlimited) | Tuned |
| `min_samples_leaf` | 1, 5 | Tuned |
| `n_estimators` | 300 | Fixed |
| `max_features` | sqrt | Fixed |
| `min_samples_split` | 2 | Fixed |
| Criterion / bootstrap | Gini / True | Fixed |
| Class weight / random seed | None / 707 | Fixed |

Table 8a gives four combinations of depth and leaf size. I select the lowest mean five-fold log loss.

#### Tune random forest


In [ ]:
forest_grid = {'model__max_depth':[10,None], 'model__min_samples_leaf':[1,5]}
search = GridSearchCV(forest_pipe,forest_grid,scoring='neg_log_loss',cv=folds,n_jobs=1,error_score='raise')
search.fit(X,y)
full_models['Random forest'] = search.best_estimator_
full_parameters['Random forest'] = search.best_params_
forest_tuning = pd.DataFrame([
    {'Model':'Random forest', 'Parameters':str(params), 'CV log loss':-score,
     'Selected':params == search.best_params_}
    for params,score in zip(search.cv_results_['params'],search.cv_results_['mean_test_score'])
])
tune_rows.extend(forest_tuning.to_dict('records'))
display(forest_tuning.drop(columns='Model').style.format(precision=4)
    .set_caption('Table 8b. Random forest tuning results'))

Table 8b selects maximum depth unlimited and minimum leaf size 5 by the lowest mean CV log loss.


#### Random forest predictions by fold


In [ ]:
for fold,(train,test) in enumerate(folds,1):
    model = clone(full_models['Random forest']).fit(X.iloc[train],y.iloc[train])
    pred = model.predict_proba(X.iloc[test])[:,1]
    oof.loc[X.index[test],'Random forest'] = pred
    fold_rows.append({'Model':'Random forest','Fold':fold,'AUROC':roc_auc_score(y.iloc[test],pred),
                      'Log loss':log_loss(y.iloc[test],pred),'Brier':brier_score_loss(y.iloc[test],pred)})

In [ ]:
forest_scores = pd.DataFrame(fold_rows).query("Model == 'Random forest'").set_index('Fold').drop(columns='Model')
forest_report = forest_scores.copy()
forest_report.loc['Mean'] = forest_scores.mean()
forest_report.loc['SD'] = forest_scores.std(ddof=1)
display(forest_report.style.format(precision=3)
    .set_caption('Table 8c. Random forest fold scores, mean and SD'))

Table 8c reports all five random forest fold scores. Mean AUROC is 0.845 (SD 0.024); mean log loss is 0.305 and Brier score is 0.093.


### 8. Compare models

Compare these three models with SAPS-I and SOFA on the same folds. Higher benchmark scores indicate higher risk. Missing or negative scores use the training mean. The receiver operating characteristic (ROC) curves use out-of-fold predictions. 

In [ ]:
assert oof[names].notna().all().all()
assert np.isfinite(oof[names].to_numpy()).all()
fold_metrics = pd.DataFrame(fold_rows).sort_values('Fold',kind='stable').reset_index(drop=True)
fold_metrics.to_csv('output/cp2_fold_metrics.csv',index=False)
tuning = pd.DataFrame(tune_rows)
tuning.to_csv('output/cp2_tuning.csv',index=False)

In [ ]:
benchmark_counts = {}
for score in ['SAPS-I','SOFA']:
    values = outcomes[score].mask(outcomes[score].lt(0))
    benchmark_counts[score] = int(values.isna().sum())
    for fold,(train,test) in enumerate(folds,1):
        pred = values.iloc[test].fillna(values.iloc[train].mean())
        assert pred.notna().all()
        oof.loc[X.index[test],score] = pred
        fold_rows.append({'Model':score,'Fold':fold,'AUROC':roc_auc_score(y.iloc[test],pred)})
all_metrics = pd.DataFrame(fold_rows)
comparison = all_metrics.pivot(index='Model',columns='Fold',values='AUROC').reindex(names+['SAPS-I','SOFA'])
comparison.columns = [f'Fold {c}' for c in comparison.columns]
comparison['Mean'] = comparison.mean(axis=1)
comparison['SD'] = comparison.iloc[:,:5].std(axis=1,ddof=1)
display(comparison.style.format(precision=3, na_rep="—")
    .set_caption('Table 9. AUROC on identical folds for all five approaches'))
observed_scores = outcomes[['SAPS-I','SOFA']].ge(0).all(axis=1)
observed_auc = {name:roc_auc_score(y[observed_scores],oof.loc[observed_scores,name])
                for name in names+['SAPS-I','SOFA']}
quality = fold_metrics.groupby('Model')[['Log loss','Brier']].mean().reindex(names)
display(quality.style.format(precision=4, na_rep="—")
    .set_caption('Table 10. Mean fold probability scores (lower is better)'))
fig,ax = plt.subplots(figsize=(7.5,6))
for name in names+['SAPS-I','SOFA']:
    fpr,tpr,_ = roc_curve(y,oof[name])
    ax.plot(fpr,tpr,label=f'{name} (pooled AUC {roc_auc_score(y,oof[name]):.3f})')
ax.plot([0,1],[0,1],ls='--',color='gray',lw=1)
ax.set(xlabel='False positive rate',ylabel='True positive rate',title='Figure 3. Out-of-fold ROC curves')
ax.legend(loc='lower right')
fig.tight_layout()
fig.savefig('output/cp2_roc.png', bbox_inches="tight")
plt.show()
plt.close(fig)


- Table 9 and Figure 3 compare all five approaches. Random forest has the highest mean AUROC (0.845); ridge has 0.842, splines 0.823.
- SAPS-I has 190 missing scores and SOFA has 144. Among the 3,733 stays with both observed, pooled AUROCs are 0.842 for ridge, 0.827 for splines, 0.844 for random forest, 0.650 for SAPS-I and 0.619 for SOFA.
- Table 10 shows the lowest mean log loss for Random forest and the lowest mean Brier score for Ridge.
- Tuning and evaluation use the same folds. These performance estimates may be optimistic.

Random forest exceeds SAPS-I by 0.200 and SOFA by 0.224 in mean AUROC. The SDs of the paired fold differences are 0.041 and 0.034, respectively. Both gains are larger than the observed fold-to-fold variation; this is a descriptive comparison, not a formal significance test.

I keep models within 0.01 of the best mean AUROC, then prefer ridge, splines and random forest in that order.

**I choose ridge.** Its mean AUROC is 0.842 (SD 0.018), 0.003 below the best. Ridge is simpler to interpret than the other two models. Its mean log loss is 0.305 and Brier score is 0.093.

### 9. Save the final model

The chosen model has been refitted on all of set-a. I save its fitted preprocessing, model and out-of-fold predictions for Checkpoint 3.


In [ ]:
best = comparison.loc[names,'Mean'].idxmax()
eligible = comparison.loc[names,'Mean'].ge(comparison.loc[best,'Mean']-.01)
chosen = next(n for n in names if eligible[n])
final_model = full_models[chosen]
joblib.dump(final_model,'output/cp2_final_model.joblib')
oof.insert(0,'In-hospital_death',y)
oof.to_csv('output/cp2_oof_predictions.csv',index_label='RecordID')
comparison.to_csv('output/cp2_comparison.csv')
quality.to_csv('output/cp2_probability_scores.csv')
loaded = joblib.load('output/cp2_final_model.joblib')
np.testing.assert_allclose(loaded.predict_proba(X.iloc[:20]),final_model.predict_proba(X.iloc[:20]))
metadata = {'input_tables':['output/set-a_wide.csv'],'seed':SEED,'selected_model':chosen,'input_columns':list(X.columns),
            'predictor_columns':predictors,'feature_imputation_means':feature_means.to_dict(),
            'raw_cleaning_rules':'checkpoint1.ipynb, Section 4','percentile_clipping':False,
            'raw_missing_values':'-1 and impossible values are NaN in the CP1 cleaned wide table',
            'model_missing_values':'Training-fold mean for numeric features; training-fold mode for categories; counts and missingness flags retained',
            'full_data_parameters':full_parameters,'benchmark_missing_counts':benchmark_counts,
            'model2_features':selected_features,
            'model2_feature_selection':'Top 15 absolute standardized ridge coefficients; C selected by shared five-fold CV; ranking fitted inside each training split',
            'cross_validation':'Five shared stratified folds; no nested CV; tuning and evaluation reuse folds, so scores may be optimistic',
            'standardization':'After imputation and encoding (and spline expansion where applicable); training-fold mean and standard deviation for all three models',
            'selection_rule':'Within 0.01 of best mean AUROC, prefer Ridge then Splines then Random forest',
            'python':platform.python_version(),'sklearn':sklearn.__version__,
            'numpy':np.__version__,'pandas':pd.__version__,
            'tuning_grids':{'Ridge':ridge_grid,'Splines':splines_grid,'Random forest':forest_grid},
            'forest_fixed_parameters':{'n_estimators':300,'max_features':'sqrt','bootstrap':True,'criterion':'gini','min_samples_split':2,'random_state':SEED},
            'elapsed_seconds':time.perf_counter()-START}
with open('output/cp2_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)


Saved files:

- `output/cp2_final_model.joblib`: fitted preprocessing and model.
- `output/cp2_oof_predictions.csv`: RecordID, outcome, fold and predictions.
- `output/cp2_metadata.json`: input columns and selected settings.

### Limitations

- Mean imputation reduces variation. Missingness and testing frequency may reflect care decisions.
- The cleaning thresholds are personal choices, not clinicians' suggestions.
- The spline model changes both the inputs and their form, so its performance difference cannot be attributed to nonlinear terms alone.
- Hyperparameters and the final model are selected using the same five folds used to report performance. The ridge C is also reused for feature selection. These CV scores may be optimistic.